# Split and Join

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
import torch
from IPython.display import SVG, display

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler.registers.register_binding import bind_register_program
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program import ProgramRecord
from flagquantum.core.ir.program.register_types import unsigned_register_type
from flagquantum.core.ir.program.signatures import Signature

## Split a word

Split consumes a packed register called `reg` and returns its bits as
an array of one-qubit words under the same name. The first element is
the most significant bit. Each bit can then be passed to an operation
separately. The physical wiring already exists, so splitting emits
no gates and allocates no work.

In [ ]:
from flagquantum.compiler.registers.component_rebindings import split_word_function

### Example Instances

In [ ]:
dtype = unsigned_register_type(4)
split = split_word_function("main", dtype)
split_signature = Signature(split.registers)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Split", split_signature, {})])))

## Join the bits

Join consumes an ordered array of one-qubit words and returns a
packed `reg` with the declared dtype. Supplying bits in their original
order recovers the original wire mapping. A different order changes
the output aliases. The relabeling emits no quantum swaps.

In [ ]:
from flagquantum.compiler.registers.component_rebindings import join_word_function

### Example Instances

In [ ]:
join = join_word_function("main", dtype)
join_signature = Signature(join.registers)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Join", join_signature, {})])))

### Append the operation between Split and Join

Bind the input using a nonconsecutive wire order. Flip the second
logical bit and join the original bit order. The result is four,
and the single X is the entire physical stream.

In [ ]:
wires = (2, 0, 3, 1)
split_bound = bind_register_program(ProgramRecord((split,)), (wires,))
join_bound = bind_register_program(ProgramRecord((join,)), split_bound.outputs)
assert split_bound.outputs == tuple((wire,) for wire in wires)
assert join_bound.outputs == (wires,)
assert split_bound.circuit.instructions == join_bound.circuit.instructions == ()
gate = Instruction("x", split_bound.outputs[1])
circuit = Circuit(4, dtype=torch.complex128)
circuit.append(FieldOperation(4, (gate,)))
ordered = circuit.state().reshape((2,) * 4).permute(wires).reshape(-1)
assert ordered[4].item() == 1 and torch.count_nonzero(ordered) == 1
assert circuit.to_ir().instructions == (gate,)